# บทที่ 4 — เตรียมข้อมูลให้พร้อมเข้าโมเดล

<sub>บทเรียนที่ 4 จาก 8 &nbsp;·&nbsp; [← บทที่ 3](03_building_labels.ipynb) · [สารบัญ](README.md) · [บทที่ 5 →](05_first_model_lstm.ipynb)</sub>

## เป้าหมายของบทนี้

เมื่อจบบทนี้คุณจะ:

- เข้าใจว่าทำไมต้องป้อนข้อมูลเป็น 'ช่วงเวลา' ไม่ใช่จุดเดียว
- สร้าง sliding window ด้วยตัวเอง
- รู้ว่าทำไมต้องแบ่งข้อมูลเป็น 3 ชุด ไม่ใช่ 2
- เข้าใจ data leakage และวิธีป้องกัน — จุดที่คนพลาดกันมากที่สุด

> **บทนี้ไม่ต้องใช้ข้อมูลดิบ** — ใช้ไฟล์ `outputs/features_cache.npz` ที่อยู่ใน repo อยู่แล้ว รันได้เลย

---

## 4.1 ปัญหา: ภาพนิ่งภาพเดียวบอกอะไรไม่ได้

ตอนนี้เรามีตาราง `(984, 56)` และคำตอบ `(984,)` แล้ว
ดูเหมือนพร้อมเทรนเลยใช่ไหม? — ยังไม่พร้อม

ลองนึกภาพว่าคุณเห็นค่า RMS = 0.12 เพียงค่าเดียว
คุณบอกได้ไหมว่าลูกปืนกำลังเสื่อม? **บอกไม่ได้** เพราะ:

- ถ้าเมื่อวาน RMS = 0.05 แล้ววันนี้ 0.12 → **กำลังแย่ลงเร็ว อันตราย**
- ถ้าเมื่อวาน RMS = 0.12 แล้ววันนี้ก็ 0.12 → **นิ่ง ปกติดี**

ค่าเดียวกันแต่ความหมายต่างกันสิ้นเชิง — สิ่งที่สำคัญคือ **แนวโน้ม** ไม่ใช่ค่า ณ จุดเดียว

**ทางแก้:** ป้อนข้อมูลย้อนหลังหลายจุดพร้อมกัน เรียกว่า **Sliding Window**

In [ ]:
# ── ตั้งค่าให้ notebook มองเห็นโค้ดใน src/ ──
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# ── หาฟอนต์ที่แสดงภาษาไทยได้ ไม่งั้นข้อความในกราฟจะกลายเป็นสี่เหลี่ยม ──
_installed = {f.name for f in fm.fontManager.ttflist}
for _f in ["Noto Sans Thai", "Leelawadee UI", "Tahoma", "TH Sarabun New", "Angsana New"]:
    if _f in _installed:
        plt.rcParams["font.family"] = _f
        plt.rcParams["axes.unicode_minus"] = False    # ฟอนต์ไทยมักไม่มีเครื่องหมายลบแบบ unicode
        print("ฟอนต์กราฟ:", _f)
        break
else:
    print("[หมายเหตุ] ไม่พบฟอนต์ไทย - ข้อความไทยในกราฟอาจแสดงเป็นสี่เหลี่ยม")
    print("           Windows/macOS มักมีอยู่แล้ว ส่วน Linux ลง: sudo apt install fonts-thai-tlwg")

print("project root:", ROOT)

In [ ]:
from src.paths import FEATURES_CACHE
from src.data_loader import compute_health_index

features = np.load(FEATURES_CACHE)["features"]
labels = compute_health_index(features, window=7)

print("features:", features.shape)
print("labels  :", labels.shape)

## 4.2 สร้าง Sliding Window เอง

**กติกา:**
- หน้าต่างกว้าง 20 จุดเวลา (20 × 10 นาที = ~3 ชั่วโมง)
- เลื่อนทีละ 1 จุด (stride = 1) เพื่อให้ได้ตัวอย่างมากที่สุด
- **คำตอบของแต่ละหน้าต่าง = Health Index ของจุดสุดท้ายในหน้าต่างนั้น**

ข้อสุดท้ายสำคัญ: เราใช้อดีต 20 จุดเพื่อทำนายสภาพ ณ ปัจจุบัน (จุดสุดท้าย)

In [ ]:
WINDOW = 20

def make_windows(feat, lab, window=WINDOW, stride=1):
    """ตัดข้อมูลเป็นหน้าต่างเลื่อน

    คืนค่า X shape (จำนวนหน้าต่าง, window, 56) และ y shape (จำนวนหน้าต่าง,)
    """
    X, y = [], []
    for start in range(0, len(feat) - window + 1, stride):
        end = start + window
        X.append(feat[start:end])   # 20 จุดเวลา x 56 features
        y.append(lab[end - 1])      # คำตอบ = จุดสุดท้ายของหน้าต่าง
    return np.array(X), np.array(y)


X, y = make_windows(features, labels)

print("X:", X.shape, " = (จำนวนหน้าต่าง, ความยาวหน้าต่าง, จำนวน features)")
print("y:", y.shape)
print()
print(f"คำนวณตรวจสอบ: {len(features)} - {WINDOW} + 1 = {len(features) - WINDOW + 1}")

### หน้าต่างหน้าตาเป็นอย่างไร

มาดูหน้าต่างเดียวแบบละเอียด

In [ ]:
i = 500   # หน้าต่างที่ 500

print(f"หน้าต่างที่ {i}")
print(f"  ครอบคลุมจุดเวลา {i} ถึง {i + WINDOW - 1}")
print(f"  = ช่วงเวลา {i*10/60:.1f} ถึง {(i+WINDOW-1)*10/60:.1f} ชั่วโมงนับจากเริ่มทดลอง")
print(f"  รูปร่างข้อมูล: {X[i].shape}")
print(f"  คำตอบ (Health Index ของจุดสุดท้าย): {y[i]:.4f}")

# พล็อตให้เห็นว่าหน้าต่างนี้ตัดมาจากตรงไหน
FEATURE_NAMES = ["RMS", "Peak", "P2P", "Crest", "Kurtosis", "Skewness",
                 "Shape", "Impulse", "Margin", "Std",
                 "BandLow", "BandMid", "BandHigh", "Centroid"]
rms_b1 = features[:, 0 * 14 + 0]   # RMS ของ Bearing 1 (ตัวที่พัง)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))

axes[0].plot(rms_b1, color="#bbb", linewidth=1)
axes[0].axvspan(i, i + WINDOW - 1, color="#1976d2", alpha=0.35)
axes[0].set_title("หน้าต่างนี้ตัดมาจากตรงไหน (แถบน้ำเงิน)")
axes[0].set_xlabel("จุดเวลา")
axes[0].set_ylabel("RMS ของ Bearing 1")

axes[1].plot(range(i, i + WINDOW), rms_b1[i:i + WINDOW], "o-", color="#1976d2")
axes[1].plot(i + WINDOW - 1, rms_b1[i + WINDOW - 1], "o", color="#2e7d32", markersize=12,
             label="จุดที่ใช้เป็นคำตอบ")
axes[1].set_title("ซูมเข้าไปดูหน้าต่างเดียว (20 จุด)")
axes[1].set_xlabel("จุดเวลา")
axes[1].legend()

plt.tight_layout()
plt.show()

## 4.3 แบ่งข้อมูล — ทำไมต้อง 3 ชุด

นักเรียนที่ทำข้อสอบที่เคยเห็นเฉลยมาแล้ว จะได้คะแนนดี
แต่คะแนนนั้นไม่ได้บอกว่าเขาเก่งจริง

โมเดลก็เหมือนกัน — ถ้าวัดผลด้วยข้อมูลชุดเดียวกับที่ใช้สอน ตัวเลขจะสวยแต่ไร้ความหมาย

| ชุด | สัดส่วน | ใช้ทำอะไร |
|---|---|---|
| **Train** | 70% | ให้โมเดลเรียนรู้ ปรับน้ำหนักภายใน |
| **Validation** | 15% | เช็คระหว่างเทรนว่าควรหยุดตอนไหน / เลือก checkpoint |
| **Test** | 15% | ข้อสอบจริง ใช้ครั้งเดียวตอนจบ |

**ทำไมต้องมี Validation แยกจาก Test?** เพราะเราใช้ Validation ตัดสินใจหลายอย่าง
(หยุดเทรนเมื่อไร เลือก checkpoint ไหน) การตัดสินใจเหล่านั้นทำให้โมเดล
"ปรับตัวเข้าหา" Validation ทางอ้อม ถ้าใช้ชุดเดียวกันวัดผลสุดท้ายด้วย
ตัวเลขก็จะดีเกินจริงอีก

### แบ่งตามเวลา หรือ สุ่มสลับ?

ประเด็นนี้สำคัญมากและมีข้อดีข้อเสียทั้งคู่ ลองดูทั้งสองแบบ

In [ ]:
n_win = len(X)

# ── แบบที่ 1: แบ่งตามลำดับเวลา ──
n_tr = int(n_win * 0.70)
n_va = int(n_win * 0.15)
seq_train = np.arange(0, n_tr)
seq_val   = np.arange(n_tr, n_tr + n_va)
seq_test  = np.arange(n_tr + n_va, n_win)

# ── แบบที่ 2: สุ่มสลับก่อนแบ่ง ──
rng = np.random.default_rng(42)
shuffled = np.arange(n_win)
rng.shuffle(shuffled)
shf_train = shuffled[:n_tr]
shf_val   = shuffled[n_tr:n_tr + n_va]
shf_test  = shuffled[n_tr + n_va:]

fig, axes = plt.subplots(2, 1, figsize=(11, 5.5))

for ax, (name, tr, va, te) in zip(axes, [
        ("แบบที่ 1: แบ่งตามลำดับเวลา", seq_train, seq_val, seq_test),
        ("แบบที่ 2: สุ่มสลับก่อนแบ่ง", shf_train, shf_val, shf_test)]):
    ax.plot(y, color="#ddd", linewidth=1)
    ax.scatter(tr, y[tr], s=4, color="#1976d2", label=f"Train ({len(tr)})")
    ax.scatter(va, y[va], s=4, color="#f57c00", label=f"Val ({len(va)})")
    ax.scatter(te, y[te], s=4, color="#2e7d32", label=f"Test ({len(te)})")
    ax.set_title(name)
    ax.set_ylabel("Health Index")
    ax.legend(loc="lower left", markerscale=3)

axes[-1].set_xlabel("ลำดับหน้าต่าง (เรียงตามเวลา)")
plt.tight_layout()
plt.show()

In [ ]:
print("ช่วงค่า Health Index ที่แต่ละชุดได้เห็น")
print("=" * 62)
for name, tr, va, te in [("แบ่งตามเวลา", seq_train, seq_val, seq_test),
                         ("สุ่มสลับ", shf_train, shf_val, shf_test)]:
    print(f"\n{name}")
    for setname, ids in [("Train", tr), ("Val", va), ("Test", te)]:
        print(f"  {setname:<6} [{y[ids].min():.4f}, {y[ids].max():.4f}]")

**นี่คือปัญหาของการแบ่งตามเวลาบนข้อมูลชุดนี้:**

Train เห็นแต่ช่วงที่ลูกปืนยังปกติ ส่วน Test เจอแต่ช่วงกำลังพัง
โมเดลที่ไม่เคยเห็นสภาพเสื่อมเลยจะทำนายช่วง Test ผิดหมด

**แต่การสุ่มสลับก็มีราคาที่ต้องจ่าย** — และนี่คือจุดที่ต้องเข้าใจให้ลึก

## 4.4 ⚠️ ราคาของการสุ่มสลับ

หน้าต่างที่อยู่ติดกันใช้ข้อมูลร่วมกันถึง **19 จาก 20 จุด**

```
หน้าต่าง 100:  [จุด 100 ... จุด 119]
หน้าต่าง 101:      [จุด 101 ... จุด 120]
                    ↑ ซ้อนกัน 19 จุด
```

พอสุ่มสลับ หน้าต่าง 100 อาจไปอยู่ Train ส่วนหน้าต่าง 101 ไปอยู่ Test
ทั้งที่มันเกือบจะเป็นข้อมูลชุดเดียวกัน

**ผลคือ Test set ไม่เป็นอิสระจาก Train อย่างแท้จริง** — ตัวเลขที่วัดได้จึงดีเกินจริง

In [ ]:
# วัดว่าปัญหานี้รุนแรงแค่ไหน
test_set = set(shf_test.tolist())
neighbours = sum(1 for i in shf_train if (i - 1) in test_set or (i + 1) in test_set)

print(f"หน้าต่างใน Train ที่มีเพื่อนบ้านติดกันอยู่ใน Test: {neighbours} จาก {len(shf_train)}")
print(f"คิดเป็น {neighbours / len(shf_train) * 100:.1f}%")
print()
print("หน้าต่างพวกนี้ซ้อนทับกัน 19/20 จุด = เกือบเป็นข้อมูลเดียวกัน")

**แล้วทำไมโปรเจกต์นี้ยังเลือกสุ่มสลับ?**

เพราะเป้าหมายคือ**เปรียบเทียบสถาปัตยกรรม 4 แบบ** ไม่ใช่วัดว่าใช้งานจริงได้แค่ไหน
ทุกโมเดลเจอเงื่อนไขเดียวกันหมด การเทียบจึงยังยุติธรรม

แต่ถ้าเป้าหมายคือ**ประเมินว่าเอาไปใช้จริงได้ไหม** ต้องแบ่งตามเวลาเท่านั้น
และต้องยอมรับว่าตัวเลขจะแย่ลงมาก

> **บทเรียน:** วิธีแบ่งข้อมูลต้องเลือกตาม*คำถามที่อยากตอบ* ไม่ใช่เลือกอันที่ให้ตัวเลขสวย
> และต้องบอกให้ชัดว่าเลือกแบบไหน เพราะอะไร

## 4.5 Normalization และกับดัก Data Leakage

feature แต่ละตัวมีหน่วยและช่วงค่าต่างกันมาก — RMS อาจอยู่ราว 0.1
ส่วน Kurtosis อาจเป็นหลักสิบ

ถ้าป้อนแบบนี้ตรง ๆ โมเดลจะให้ความสำคัญกับ feature ที่มีตัวเลขใหญ่มากเกินไป
จึงต้องปรับให้อยู่สเกลใกล้เคียงกันด้วย `StandardScaler`:

```
ค่าใหม่ = (ค่าเดิม − ค่าเฉลี่ย) / ส่วนเบี่ยงเบนมาตรฐาน
```

**คำถามสำคัญ: ค่าเฉลี่ยกับส่วนเบี่ยงเบนนี้ ควรคำนวณจากข้อมูลชุดไหน?**

In [ ]:
from sklearn.preprocessing import StandardScaler

# ── วิธีผิด: fit จากข้อมูลทั้งหมด ──
scaler_wrong = StandardScaler().fit(features)

# ── วิธีถูก: fit เฉพาะจุดเวลาที่อยู่ในหน้าต่างของ Train ──
train_rows = np.unique(np.concatenate([np.arange(s, s + WINDOW) for s in shf_train]))
scaler_right = StandardScaler().fit(features[train_rows])

print(f"วิธีผิด : fit จาก {len(features)} จุดเวลา (ทั้งหมด)")
print(f"วิธีถูก : fit จาก {len(train_rows)} จุดเวลา (เฉพาะที่อยู่ใน train windows)")
print()
print("ค่าเฉลี่ยของ RMS Bearing 3 ที่คำนวณได้:")
print(f"  วิธีผิด : {scaler_wrong.mean_[28]:.6f}")
print(f"  วิธีถูก : {scaler_right.mean_[28]:.6f}")

**ทำไมวิธีแรกถึงผิด?**

ค่าเฉลี่ยกับส่วนเบี่ยงเบนที่คำนวณจากข้อมูลทั้งหมด **มีข้อมูลของ Test set ปนอยู่**
เท่ากับโมเดลได้แอบรู้บางอย่างเกี่ยวกับข้อสอบล่วงหน้า

เรียกว่า **Data Leakage** — เป็นข้อผิดพลาดที่พบบ่อยที่สุดใน ML
และร้ายตรงที่มัน**ไม่ทำให้โค้ดพัง** แค่ทำให้ผลดูดีเกินจริงเงียบ ๆ

> ในชุดข้อมูลนี้ตัวเลขต่างกันน้อยมาก เพราะ train windows ครอบคลุมเกือบทุกจุดเวลาอยู่แล้ว
> แต่**หลักการต้องถูกเสมอ** เพราะในงานอื่นความต่างอาจมหาศาล

## 4.6 ใช้ของจริงจาก `src/dataset.py`

โค้ดจริงทำทุกขั้นตอนข้างต้นให้ในฟังก์ชันเดียว

In [ ]:
from src.dataset import split_dataset

train_loader, val_loader, test_loader, scaler = split_dataset(
    features, labels,
    window_size=WINDOW,
    batch_size=32,
    shuffle_split=True,
    random_seed=42,
)

xb, yb = next(iter(train_loader))
print()
print("ข้อมูล 1 batch ที่โมเดลจะได้รับ:")
print("  x:", tuple(xb.shape), " = (batch, ความยาวหน้าต่าง, features)")
print("  y:", tuple(yb.shape))

## 🔧 ลองแก้ดู — ทดลองกับการเตรียมข้อมูล


1. เปลี่ยน `WINDOW = 20` เป็น `5` และ `60` แล้วดูว่าจำนวนหน้าต่างเปลี่ยนไปเท่าไร
   หน้าต่างสั้นเกินไปจะเสียอะไร? ยาวเกินไปล่ะ?
2. เปลี่ยน `random_seed=42` เป็นเลขอื่น แล้วดูว่าช่วงค่าของแต่ละชุดเปลี่ยนไหม
   ถ้าเปลี่ยนมาก แปลว่าผลการทดลองไวต่อการสุ่มแค่ไหน?
3. ลองเปลี่ยน `shuffle_split=True` เป็น `False` แล้วเก็บไว้ก่อน —
   บทที่ 5 จะให้ลองเทรนด้วยการแบ่งแบบนี้เพื่อดูว่าผลต่างกันแค่ไหน

## ❓ เช็คความเข้าใจ

**1. ทำไมคำตอบของหน้าต่างถึงเป็นจุดสุดท้าย ไม่ใช่จุดแรกหรือจุดกลาง?**

<details>
<summary>ดูเฉลย</summary>

เพราะเราต้องการทำนายสภาพ ณ ปัจจุบันจากข้อมูลย้อนหลัง ถ้าใช้จุดแรกเป็นคำตอบ เท่ากับใช้ข้อมูลอนาคต 19 จุดมาทำนายอดีต ซึ่งใช้งานจริงไม่ได้ ถ้าใช้จุดกลางก็ยังใช้ข้อมูลอนาคตอยู่ครึ่งหนึ่ง

</details>

**2. ถ้าเราแบ่งข้อมูลก่อนแล้วค่อยตัดหน้าต่าง จะแก้ปัญหาหน้าต่างซ้อนทับได้ไหม?**

<details>
<summary>ดูเฉลย</summary>

ได้ และเป็นวิธีที่ถูกต้องกว่า ถ้าแบ่งช่วงเวลาออกจากกันก่อนแล้วค่อยตัดหน้าต่างภายในแต่ละช่วง หน้าต่างจาก Train กับ Test จะไม่มีทางซ้อนทับกันเลย (ยกเว้นตรงรอยต่อไม่กี่หน้าต่าง ซึ่งตัดทิ้งได้)

</details>

**3. Data leakage แบบไหนที่อันตรายที่สุด เพราะอะไร?**

<details>
<summary>ดูเฉลย</summary>

แบบที่ไม่ทำให้โค้ดพังและตัวเลขดูดีขึ้นเล็กน้อย เพราะจะไม่มีใครสังเกตเห็น แล้วผลที่รายงานไปก็จะดีเกินจริงตลอด ต่างจาก leakage รุนแรงที่ทำให้ได้ความแม่นยำ 99.9% ซึ่งผิดปกติจนคนสงสัยเอง

</details>

---

## สรุปบทนี้

- Sliding window ทำให้โมเดลเห็นแนวโน้ม ไม่ใช่แค่ค่า ณ จุดเดียว
- แบ่ง 3 ชุด: Train สอน / Val ตัดสินใจระหว่างเทรน / Test วัดผลครั้งเดียว
- สุ่มสลับแก้ปัญหาการกระจายของ label แต่ทำให้ Test ไม่เป็นอิสระเต็มที่
- Normalization ต้อง fit จาก Train เท่านั้น ไม่งั้นเกิด data leakage

[← บทที่ 3](03_building_labels.ipynb) &nbsp;·&nbsp; [สารบัญ](README.md) &nbsp;·&nbsp; **[บทที่ 5 — โมเดลแรก →](05_first_model_lstm.ipynb)**